# Week 1 · Day 2 — Lab 3
## Indexing, Selection & the View/Copy Trap

Selecting the right elements is most of data work: filter the failures, sample a
batch, look up labels, slice a window. NumPy gives you three selection tools —
**basic slicing**, **boolean masking**, and **fancy (integer-array) indexing** —
and they differ in one safety-critical way: *some return a view of the original
buffer, others return an independent copy.* Getting this wrong is one of the most
common silent bugs in NumPy code. This lab makes the rules muscle memory.

You'll work with 240 composite eval scores for model responses, plus a fixed
shuffle index and per-response category codes.

### Learning objectives
1. Slice 1-D and 2-D arrays with `start:stop:step`.
2. Tell **views** (basic slices) from **copies** (boolean / fancy) and prove it with `np.shares_memory`.
3. Build boolean masks with compound conditions (`&`, `|`, `~`, and parentheses).
4. Use fancy indexing for **shuffling**, **batch sampling**, and **lookup tables**.
5. Defuse the "a slice is a copy" production bug by copying at function boundaries.

### Time budget — ~70 min
| Segment | Time |
|---|---|
| Framing & objectives | 5 min |
| **A.** Slicing 1-D & 2-D | 10 min |
| **B.** Views vs. copies | 12 min |
| **C.** Boolean masking | 15 min |
| **D.** Fancy indexing | 15 min |
| **E.** The slice-is-a-copy bug | 10 min |
| Wrap-up + stretch | 3 min |

### Files you need (in `data/`)
- `lab3_eval_scores.npy` — shape (240,), `float64`, scores in [0, 1].
- `lab3_shuffle_index.npy` — shape (240,), `int64`, a fixed permutation of 0..239.
- `lab3_category_codes.npy` — shape (240,), `int8`, a category code 0..4 per response.


In [ ]:
import numpy as np
from pathlib import Path

print("NumPy", np.__version__)   # target curriculum: NumPy 2.x on Python 3.13

# Solution is different here because of folder structure

DATA = Path("../data")
if not DATA.exists():
    DATA = Path(".")

def check(label, predicate):
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

# 240 composite eval scores in [0, 1] for 240 model responses.
scores = np.load(DATA / "lab3_eval_scores.npy")
scores_orig = scores.copy()                       # pristine reference for Part E
shuffle_idx = np.load(DATA / "lab3_shuffle_index.npy")   # a fixed permutation of 0..239
category_codes = np.load(DATA / "lab3_category_codes.npy")  # int code 0..4 per response

# Small vocabulary the codes index into (lookup-table fancy indexing in Part D).
CATEGORIES = np.array(["factual", "creative", "code", "refusal", "summary"])

print("scores:", scores.shape, scores.dtype, "| range",
      scores.min().round(3), "->", scores.max().round(3))
print("shuffle_idx:", shuffle_idx.shape, shuffle_idx.dtype)
print("category_codes:", category_codes.shape, category_codes.dtype)

## Part A — Slicing  *(guided)*

`a[start:stop:step]` — `stop` is exclusive; omit any part to default to
begin / end / 1. A negative step reverses. For 2-D, give one slice per axis.


In [ ]:
a = np.arange(10)
print("a[2:7]   ->", a[2:7])     # 2..6
print("a[::2]   ->", a[::2])     # every other
print("a[::-1]  ->", a[::-1])    # reversed

grid = scores.reshape(24, 10)    # a 24x10 view of the 240 scores
print("grid shape:", grid.shape)
print("center 2x2 (rows 1:3, cols 4:6):\n", grid[1:3, 4:6].round(3))

### Exercise A1 — Three slices
From the 1-D `scores` array, produce:
- `first_50` — the first 50 scores,
- `every_10th` — every 10th score starting at index 0,
- `reversed_scores` — all scores in reverse order.


In [ ]:
first_50 = scores[:50]
every_10th = scores[::10]
reversed_scores = scores[::-1]
print("first_50:", first_50.shape, "| every_10th:", every_10th.shape)
print("reversed first == original last:", reversed_scores[0] == scores[-1])

In [ ]:
check("A1: first_50 has 50 elements", lambda: first_50.shape == (50,))
check("A1: every_10th has 24 elements", lambda: every_10th.shape == (24,))
check("A1: reversed_scores reverses the array",
      lambda: np.array_equal(reversed_scores, scores[::-1]))

## Part B — Views vs. copies

**Basic slices are views** — they share the original buffer, so writing through
them edits the original. `.copy()` detaches. This is *memory semantics*, not a
style choice.


In [ ]:
demo = np.arange(10)
window = demo[2:6]        # VIEW
window[0] = 999
print("editing the slice changed the original:", demo)   # demo[2] == 999
print("shares_memory:", np.shares_memory(demo, window))

### Exercise B1 — View vs. copy, proven
Take a slice `head = scores[:20]` and confirm it is a **view**. Then make
`head_copy = scores[:20].copy()` and confirm it is **independent**. Record the
two `np.shares_memory` booleans.


In [ ]:
head = scores[:20]
head_copy = scores[:20].copy()
head_is_view = np.shares_memory(scores, head)
copy_is_view = np.shares_memory(scores, head_copy)
print("slice is a view :", head_is_view)
print("copy is a view  :", copy_is_view)

In [ ]:
check("B1: basic slice IS a view", lambda: head_is_view is True)
check("B1: .copy() is NOT a view", lambda: copy_is_view is False)

🧑‍🏫 **Instructor note — B1.** Expected: view=True, copy=False. Drive home the
mental rule before Part E: *basic slice → view; `.copy()` → independent.* The
upcoming bug lives entirely in this distinction.


## Part C — Boolean masking

A comparison produces a boolean array; indexing with it selects the `True`
elements. **Boolean indexing always returns a copy.** Compound conditions must
use the bitwise operators `&` (and), `|` (or), `~` (not) — and each condition
needs **parentheses**, because `&`/`|` bind tighter than comparisons.


In [ ]:
s = np.array([72, 85, 91, 60, 78, 95, 55])
print("mask >= 80   ->", s >= 80)
print("select       ->", s[s >= 80])
print("70 <= s < 90 ->", s[(s >= 70) & (s < 90)])   # note the parentheses!

### Exercise C1 — Mid-band responses
Select every score in the half-open band `[0.6, 0.9)` into `mid_band`, and set
`n_mid` to how many there are. Use a single compound mask.


In [ ]:
mask = (scores >= 0.6) & (scores < 0.9)
mid_band = scores[mask]
n_mid = int(mask.sum())
print("count in [0.6, 0.9):", n_mid)
print("is a copy (not a view):", not np.shares_memory(scores, mid_band))

In [ ]:
check("C1: mid_band values all in [0.6, 0.9)",
      lambda: bool(((mid_band >= 0.6) & (mid_band < 0.9)).all()))
check("C1: n_mid equals the mask count",
      lambda: n_mid == int(((scores >= 0.6) & (scores < 0.9)).sum()))
check("C1: boolean indexing returned a COPY",
      lambda: not np.shares_memory(scores, mid_band))

### Exercise C2 — The tails (OR)
Select the *extreme* responses — scores `>= 0.95` **or** `< 0.3` — into `tails`.


In [ ]:
tails = scores[(scores >= 0.95) | (scores < 0.3)]
print("tail count:", tails.shape[0])

In [ ]:
check("C2: every tail value is in an extreme band",
      lambda: bool(((tails >= 0.95) | (tails < 0.3)).all()))

🧑‍🏫 **Instructor note — C.** The classic failure here is `scores >= 0.6 &
scores < 0.9` without parentheses → `&` binds first → `TypeError` /
`ValueError`. If a student hits it, that's the lesson, not a detour. Also note
`and`/`or` (Python keywords) don't work element-wise — must be `&`/`|`.


## Part D — Fancy (integer-array) indexing

Indexing with an **array of integers** selects those positions, in that order,
duplicates allowed. **Fancy indexing always returns a copy.** It's how you
implement dataset shuffling, batch sampling, and lookup tables.


In [ ]:
vals = np.array([10, 20, 30, 40, 50, 60])
print(vals[[0, 2, 4]])        # [10 30 50]
print(vals[[3, 0, 3, 5]])     # arbitrary order + duplicates -> [40 10 40 60]

### Exercise D1 — Shuffle by index
`shuffle_idx` is a fixed permutation of 0..239. Use it to produce a shuffled
copy of the scores, `shuffled`. (This is reproducible *without* the RNG — that
arrives in Lab 5.)


In [ ]:
shuffled = scores[shuffle_idx]
print("first 5 shuffled:", shuffled[:5].round(3))
print("same multiset as original:", np.array_equal(np.sort(shuffled), np.sort(scores)))

In [ ]:
check("D1: shuffled is a permutation of scores",
      lambda: np.array_equal(np.sort(shuffled), np.sort(scores)))
check("D1: shuffled is a copy", lambda: not np.shares_memory(scores, shuffled))

### Exercise D2 — Sample a mini-batch
Take the **first 32** indices of `shuffle_idx` as a batch selector, then gather
those scores into `batch` (shape `(32,)`).


In [ ]:
batch_idx = shuffle_idx[:32]
batch = scores[batch_idx]
print("batch shape:", batch.shape)

In [ ]:
check("D2: batch shape (32,)", lambda: batch.shape == (32,))
check("D2: batch matches scores[batch_idx]",
      lambda: np.array_equal(batch, scores[shuffle_idx[:32]]))

### Exercise D3 — Lookup table
Turn the integer `category_codes` into human-readable labels by indexing the
`CATEGORIES` vocabulary array with them: `labels = CATEGORIES[category_codes]`.
This is exactly how token-id → token lookups work.


In [ ]:
labels = CATEGORIES[category_codes]
print("first 8 labels:", labels[:8])
uniq, counts = np.unique(labels, return_counts=True)
print(dict(zip(uniq.tolist(), counts.tolist())))

In [ ]:
check("D3: labels shape (240,)", lambda: labels.shape == (240,))
check("D3: every label came from the vocabulary",
      lambda: set(np.unique(labels).tolist()) <= set(CATEGORIES.tolist()))
check("D3: label 0 matches its code",
      lambda: labels[0] == CATEGORIES[category_codes[0]])

🧑‍🏫 **Instructor note — D.** D1 multiset-equality is the right way to assert a
shuffle (don't assert order). D3 is the conceptual seed for embedding lookups and
tokenizer vocab tables in Weeks 3–5: an integer array indexing a table is the
whole trick.


## Part E — The slice-is-a-copy bug (and the fix)

Here's the bug that bites everyone once. A function mutates its input array. The
caller passed a **slice** (a view), so the mutation silently corrupts the
caller's original data.


In [ ]:
def preprocess_unsafe(batch):
    batch[batch < 0.5] = 0.0       # mutates in place
    return batch

demo = scores_orig.copy()
sample = demo[10:20]               # a VIEW into demo
_ = preprocess_unsafe(sample)
print("demo[10:20] was modified by the function call!")
print("changed:", not np.array_equal(demo[10:20], scores_orig[10:20]))

### Exercise E1 — Make it safe
Write `preprocess_safe(batch)` that does the same clipping but **copies at the
function boundary** first, so the caller's array is never touched. The check
runs it on a fresh view and confirms the source array is unchanged.


In [ ]:
def preprocess_safe(batch):
    batch = batch.copy()           # detach from the caller's buffer
    batch[batch < 0.5] = 0.0
    return batch

In [ ]:
_demo = scores_orig.copy()
_view = _demo[10:20]
_out = preprocess_safe(_view)
check("E1: function returned a clipped array",
      lambda: _out is not None and float(_out.min()) >= 0.0)
check("E1: caller's array is UNCHANGED",
      lambda: np.array_equal(_demo, scores_orig))

🧑‍🏫 **Instructor note — E1.** Expected: both PASS. The rule for the room:
*copy defensively at function boundaries unless in-place mutation is the
explicit, documented contract.* This exact failure mode resurfaces in Week 2
data-cleaning code and Week 7 pipeline steps — calling it out now saves hours.


## Stretch goals *(for fast finishers)*

**S1 — Coordinate pairs.** From `grid` (24×10), pull the three elements at
(row, col) pairs (0, 0), (5, 9), (23, 4) in one fancy-index expression returning
a length-3 array.

**S2 — Indices, not values.** Use `np.where(scores >= 0.95)` (single-argument
form) to get the *positions* of the top responses, and confirm those positions
really do select scores ≥ 0.95.


In [ ]:
corner_vals = grid[[0, 5, 23], [0, 9, 4]]
print("S1 corner_vals:", corner_vals.round(3))

top_positions = np.where(scores >= 0.95)[0]
print("S2 #positions:", top_positions.size)
print("S2 all selected >= 0.95:", bool((scores[top_positions] >= 0.95).all()))

In [ ]:
check("S1: three coordinate values selected", lambda: corner_vals.shape == (3,))
check("S2: positions select only scores >= 0.95",
      lambda: bool((scores[top_positions] >= 0.95).all()))

## Wrap-up — what you can now do

- Slice 1-D and 2-D arrays and reverse with a negative step.
- State the rule cold: **basic slice → view; boolean & fancy → copy**, and prove it with `np.shares_memory`.
- Build compound boolean masks with `& | ~` and the right parentheses.
- Shuffle, sample batches, and run lookup tables with fancy indexing.
- Copy defensively at function boundaries to avoid corrupting a caller's data.

**Next:** Lab 4 — collapsing arrays with axis-aware **reductions**, NaN-safe
stats, and sorting/ranking.
